<a href="https://colab.research.google.com/github/shin-noda/leetcode-problemset-97/blob/main/Problem3213.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from collections import deque

class TrieNode:
    def __init__(self):
        # Flattening transitions: children will store direct jumps
        self.children = {}
        self.fail = None

        # Dict mapping {word_length: minimum_cost} to auto-deduplicate
        self.output = {}


class Solution:
    def minimumCost(self, target, words, costs):
        # Use Trie + Aho-Corasick + DP

        root = TrieNode()

        # Build the Trie
        for word, cost in zip(words, costs):
            curr = root

            for char in word:
                if char not in curr.children:
                    curr.children[char] = TrieNode()

                curr = curr.children[char]

            # Store / Optimze directly inline on the node
            w_len = len(word)
            curr.output[w_len] = min(curr.output.get(w_len, float('inf')), cost)

        # Build Failure Links & Automaton Transitions via BFS
        queue = deque()

        # Initialize first level
        for char, child in root.children.items():
            child.fail = root
            queue.append(child)

        while queue:
            curr = queue.popleft()

            # Deduplicate and propagate output matches from failure link
            if curr.fail:
                for length, cost in curr.fail.output.items():
                    curr.output[length] = min(curr.output.get(length, float('inf')), cost)

            # Pre-calculate transitions for all possible 26 characters
            for char, child in curr.children.items():
                # Find failure link for the child
                fail_node = curr.fail

                while fail_node and char not in fail_node.children:
                    fail_node = fail_node.fail

                if fail_node:
                    child.fail = fail_node.children[char]
                else:
                    child.fail = root

                queue.append(child)

        # Dynamic Programming Processing
        n = len(target)
        dp = [float('inf')] * (n + 1)
        dp[0] = 0

        # 'curr' is the longest suffix of the characters we've read so far
        # that is also a prefix in the trie.
        curr = root
        for i in range(1, n + 1):
            char = target[i - 1]

            # Aho-Corasick automaton jump: O(1) step
            while curr and char not in curr.children:
                curr = curr.fail

            if curr:
                curr = curr.children[char]

            else:
                curr = root

            # Check only unique, fully optimized word lengths
            if curr:
                for length, cost in curr.output.items():
                    if dp[i - length] != float('inf'):
                        if dp[i - length] + cost < dp[i]:
                            dp[i] = dp[i - length] + cost

        if dp[n] == float('inf'):
            return -1

        return dp[n]

In [2]:
solution = Solution()

In [3]:
target = "abcdef"
words = ["abdef","abc","d","def","ef"]
costs = [100,1,1,10,5]

print(solution.minimumCost(target, words, costs))

7


In [4]:
target = "aaaa"
words = ["z","zz","zzz"]
costs = [1,10,100]

print(solution.minimumCost(target, words, costs))

-1
